In [1]:
import depthai as dai
import numpy as np
import json
import ast



In [2]:
def build_homogeneous(rotation_matrix, translation_vector):
    T_camera_to_base_effector = np.eye(4)
    T_camera_to_base_effector[:3, :3] = rotation_matrix
    T_camera_to_base_effector[:3, 3] = translation_vector.reshape(3)
    return T_camera_to_base_effector

def convert_coordinates(x ,y ,z, homogeneous_matrix): # X Y Z coordinates that should be translated into robot frame coordinates
    obj_camera_coordinates = np.array([x, y, z])
    obj_camera_coordinates_homo = np.append(obj_camera_coordinates, [1])  # Convert object coordinates to homogeneous coordinates
    obj_base_effector_coordinates_homo = homogeneous_matrix.dot(obj_camera_coordinates_homo)
    obj_base_coordinates = obj_base_effector_coordinates_homo[:3]  
    
    #return list(map(int, obj_base_coordinates)) # Uncomment this line if you want to send integers instead of floats
    return np.around(obj_base_coordinates,2).tolist() # Use this to get a list of new coordinates, chane the number to get the number of decimal numbers

def estimate_rigid_transform(camera_points, robot_points):
    cam = np.asarray(camera_points, dtype=np.float64)
    rob = np.asarray(robot_points,  dtype=np.float64)
    assert cam.shape == rob.shape and cam.shape[1] == 3 and cam.shape[0] >= 3, "Value error, differing amount of coordinates, Camera:" + str(len(cam)) + " Robot:"+ str(len(rob))
    

    camera_centroid = cam.mean(axis=0)
    robot_centroid  = rob.mean(axis=0)
    camera_centered = cam - camera_centroid
    robot_centered  = rob - robot_centroid

    cross_covariance = camera_centered.T @ robot_centered
    U, s, Vt = np.linalg.svd(cross_covariance)
    V = Vt.T

    # Ensure of proper rotation (det=+1)
    det_correction = np.sign(np.linalg.det(V @ U.T))
    rotation_matrix = V @ np.diag([1.0, 1.0, det_correction]) @ U.T

    translation_vector = robot_centroid - rotation_matrix @ camera_centroid
    return rotation_matrix, translation_vector

def extract_data(file):
    
    float_list=[]
    with open(file, "r") as f:
        lines = f.readlines()
        for i in lines:
            x = json.loads(i)
            float_list.append(x)
    return list(float_list)

In [3]:
cam_coords = extract_data("saved_coordinates.txt")
robot_coords = extract_data("robo_coords.txt")

print(cam_coords)
print(robot_coords)



[[18.804306030273438, -8.065057754516602, 775.48779296875], [-253.22769165039062, 37.41748046875, 981.2294921875], [-227.116943359375, 149.2969970703125, 1285.5679931640625], [24.686628341674805, 39.52834701538086, 950.2024536132812], [-72.41410064697266, 160.78977966308594, 1306.5283203125], [-209.0472869873047, 39.44023513793945, 989.305419921875], [65.744384765625, 32.89694595336914, 903.7639770507812], [-150.45704650878906, 61.27862548828125, 1010.0891723632812], [-35.62618637084961, 162.66688537597656, 1285.5679931640625], [-23.708532333374023, 7.41449499130249, 855.5203247070312]]
[[394.163, 360.014, 45.2468], [598.309, 95.9641, 44.5703], [447.773, -180.739, 46.378], [318.487, 198.147, 37.1674], [280.026, -154.354, 38.4586], [548.93, 95.9352, 39.5118], [301.78, 265.922, 42.8517], [479.935, 73.9366, 44.6034], [252.326, -131.682, 42.2974], [411, 267.247, 37.757]]


In [4]:
R, t = estimate_rigid_transform(cam_coords, robot_coords)
print("R =\n", R)
print("t =", t)
homogeneous = build_homogeneous(R,t)
print(homogeneous)
print(convert_coordinates(100,100,100, homogeneous))

R =
 [[-0.93480345 -0.16182031 -0.31615929]
 [ 0.3513136  -0.29054895 -0.89003374]
 [ 0.05216579 -0.94307767  0.32845583]]
t = [ 660.7177316  1059.75511264 -228.96709574]
[[-9.34803448e-01 -1.61820315e-01 -3.16159293e-01  6.60717732e+02]
 [ 3.51313597e-01 -2.90548946e-01 -8.90033745e-01  1.05975511e+03]
 [ 5.21657914e-02 -9.43077672e-01  3.28455833e-01 -2.28967096e+02]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
[519.44, 976.83, -285.21]


In [5]:
def parse_file(path):
    R_list = []
    t_list = []

    with open(path, "r") as f:
        block = []
        for line in f:
            line = line.strip()
            if not line:
                continue
            block.append(line)

            if len(block) == 4:
                R = []
                for i in range(3):
                    R.append([float(x) for x in block[i].replace("[","").replace("]","").split()])
                R = np.array(R)
                t = np.array([float(x) for x in block[3].replace("[","").replace("]","").split()])
                R_list.append(R)
                t_list.append(t)
                block = []

    return np.array(R_list), np.array(t_list)


def average_rotations(Rs):
    M = np.zeros((3, 3))
    for R in Rs:
        M += R
    M /= len(Rs)
    
    U, _, Vt = np.linalg.svd(M)
    R_avg = U @ Vt

    return R_avg


def average_translations(ts):
    return np.mean(ts, axis=0)


def save_result(path, R_avg, t_avg):
    with open(path, "w") as f:
        f.write(str(R_avg) + "\n")
        f.write(str(t_avg) + "\n")



input_file = "RT.txt"
output_file = "AveragedOutput.txt"

R_list, t_list = parse_file(input_file)

R_avg = average_rotations(R_list)
t_avg = average_translations(t_list)

save_result(output_file, R_avg, t_avg)

print("Average Rotation:\n", R_avg)
print("Average Translation:\n", t_avg)


Average Rotation:
 [[-0.92846361 -0.17297599 -0.32868622]
 [ 0.36601089 -0.27557182 -0.88887355]
 [ 0.06317712 -0.94558949  0.31916951]]
Average Translation:
 [ 677.76346589 1068.38548463 -216.59972571]


In [6]:
import numpy as np

def build_homogeneous(rotation_matrix, translation_vector):
    T_camera_to_base_effector = np.eye(4)
    T_camera_to_base_effector[:3, :3] = rotation_matrix
    T_camera_to_base_effector[:3, 3] = translation_vector.reshape(3)
    return T_camera_to_base_effector

def load_rt_from_txt(path):
    with open(path, 'r') as f:
        text = f.read().strip()
    blocks = [b for b in text.split('\n\n') if b.strip()]

    rotations = []
    translations = []

    for block in blocks:
        lines = [ln for ln in block.splitlines() if ln.strip()]
        R_rows = []
        for ln in lines[:3]:
            nums = [float(x) for x in ln.replace('[', '').replace(']', '').split()]
            R_rows.append(nums)
        R = np.array(R_rows) 

        t_line = lines[3]
        t = np.array([float(x) for x in t_line.replace('[', '').replace(']', '').split()])  # (3,)

        rotations.append(R)
        translations.append(t)

    return np.array(rotations), np.array(translations)

# Example usage
rotations, translations= load_rt_from_txt("RT.txt")
homo = build_homogeneous(rotations,translations)



ValueError: could not broadcast input array from shape (5,3,3) into shape (3,3)

In [1]:
import numpy as np

def parse_file(path):
    R_list = []
    t_list = []

    with open(path, "r") as f:
        block = []
        for line in f:
            line = line.strip()
            if not line:
                continue
            block.append(line)

            if len(block) == 4:
                R = []
                for i in range(3):
                    R.append([float(x) for x in block[i].replace("[","").replace("]","").split()])
                R = np.array(R)
                t = np.array([float(x) for x in block[3].replace("[","").replace("]","").split()])
                R_list.append(R)
                t_list.append(t)
                block = []

    return np.array(R_list), np.array(t_list)


def average_rotations(Rs):
    M = np.zeros((3, 3))
    for R in Rs:
        M += R
    M /= len(Rs)

    # Project back to closest proper rotation matrix
    U, _, Vt = np.linalg.svd(M)
    R_avg = U @ Vt
    return R_avg


def average_translations(ts):
    return np.mean(ts, axis=0)


def save_result(path, R_avg, t_avg):
    with open(path, "w") as f:
        # Same style as your input: 3 lines for R, 1 line for t
        for row in R_avg:
            f.write("[" + " ".join(f"{v:.8f}" for v in row) + "]\n")
        f.write("[" + " ".join(f"{v:.8f}" for v in t_avg) + "]\n")


def build_homogeneous(rotation_matrix, translation_vector):
    T_camera_to_base_effector = np.eye(4)
    T_camera_to_base_effector[:3, :3] = rotation_matrix
    T_camera_to_base_effector[:3, 3] = translation_vector.reshape(3)
    return T_camera_to_base_effector

#Hela denna delen#########
input_file = "RT.txt"
output_file = "AveragedOutput.txt"

R_list, t_list = parse_file(input_file)


R_avg = average_rotations(R_list)
t_avg = average_translations(t_list)

homogeneous2 = build_homogeneous(R_avg, t_avg)
save_result(output_file, R_avg, t_avg)
##################################



print("Average Rotation:\n", R_avg)
print("Average Translation:\n", t_avg)
print("Average Homogeneous matrix:\n", homogeneous2)


Average Rotation:
 [[-0.92846361 -0.17297599 -0.32868622]
 [ 0.36601089 -0.27557182 -0.88887355]
 [ 0.06317712 -0.94558949  0.31916951]]
Average Translation:
 [ 677.76346589 1068.38548463 -216.59972571]
Average Homogeneous matrix:
 [[-9.28463611e-01 -1.72975989e-01 -3.28686219e-01  6.77763466e+02]
 [ 3.66010894e-01 -2.75571824e-01 -8.88873554e-01  1.06838548e+03]
 [ 6.31771207e-02 -9.45589487e-01  3.19169507e-01 -2.16599726e+02]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


In [54]:
R_list, t_list = parse_file(input_file)

# print("R_list:", R_list)
# print("\n")
# print("t_list:", t_list)

# print("R avg:", R_avg)
# print("t avg:", t_avg)

# n = len(R_list)
# diffR, diffT = [[0,0,0],[0,0,0],[0,0,0]], [0,0,0]
# for r in R_list:
#     tempr = r - R_avg
#     diffR = diffR + tempr
#     #print(temp)


# for t in t_list:
#     tempt = t - t_avg
#     diffT = diffT + tempt
#     #print(temp)

# sigma1 = np.sqrt((diffR/n)**2)
# sigma2 = np.sqrt((diffT/n)**2)
# print("sigma R:\n", sigma1)
# print("sigma t:\n", sigma2)

In [58]:
#STANDARD DEVIATION
def standard_deviation(Rotations, Rotation_avg, Translations, Translation_avg):

    sigma_R = np.sqrt(np.mean((Rotations - Rotation_avg)**2, axis=0))
    sigma_t = np.sqrt(np.mean((Translations - Translation_avg)**2, axis=0))

    # print("Sigma R:\n", sigma_R)
    # print("Sigma t:\n", sigma_t)
    return sigma_R, sigma_t

stdR, stdT =standard_deviation(R_list, R_avg, t_list, t_avg)
print(stdR, stdT)

[[0.00766969 0.06158159 0.03544963]
 [0.0171613  0.01468592 0.00991067]
 [0.06796027 0.01206719 0.01704742]] [36.54304179  7.97728101 14.7309668 ]
